# MiniCPM5-1B address repair SFT run

Trains a LoRA adapter (r16) on 5,000 evidence-preserving targets and
picks the checkpoint on honesty metrics (review precision, 1-damage, F1),
never recall alone. Checkpoints go to Drive; git keeps hashes.

The published 2026-09-10 T4 run is `evals/sft-v1/` plus
`configs/train_t4_run.yaml` (commit `911cb9f`, UUID-only splits).
A new run of this notebook uses the current pair-grouped split code.
Commands: `docs/REPRO.md`.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!git clone https://github.com/TMFNK/local-slm-de-address-repair.git
%cd local-slm-de-address-repair
!pip install -q uv
!uv sync


Data: either copy the two slice CSVs from Drive (the T4 path) or
download the Zenodo archive. Hashes must match `configs/data.yaml`:

- dirty `2ef8ab1424c8c357af1aeaecef12b376cb7f20616437035f2753d0b2f2ac86ef`
- clean `78c852e3a5680a7460732b67ab87d82484a718297190fa8852af73a80817a192`


In [ ]:
!mkdir -p data/raw
!cp "/content/drive/MyDrive/colab-data/dirty.csv" data/raw/dirty.csv
!cp "/content/drive/MyDrive/colab-data/clean.csv" data/raw/clean.csv
!sha256sum data/raw/dirty.csv data/raw/clean.csv


Optional Zenodo path (skip if the Drive copies above already hashed).
Expect `AddressTable.zip` at 7,263,043,993 bytes. On repeated 504s,
stay on the Drive copies.


In [ ]:
# !mkdir -p data/raw
# !wget -c -O data/raw/AddressTable.zip "https://zenodo.org/records/20841898/files/AddressTable.zip?download=1"
# !unzip -o data/raw/AddressTable.zip 'AddressTable/de_slice/named/*' -d data/raw/_addr_tmp
# !cp data/raw/_addr_tmp/AddressTable/de_slice/named/slice_de_dirty.csv data/raw/dirty.csv
# !cp data/raw/_addr_tmp/AddressTable/de_slice/named/slice_de_clean.csv data/raw/clean.csv


In [ ]:
!uv run python -c "import csv; [print(f, sum(1 for _ in open(f)) - 1, 'rows') for f in ('data/raw/dirty.csv', 'data/raw/clean.csv')]"


In [ ]:
# Drive copies, no zip on disk (T4 run):
!uv run python scripts/prepare_data.py --config configs/data.yaml --skip-archive --force
# Full archive on disk, use this instead:
# !uv run python scripts/prepare_data.py --config configs/data.yaml --force


Hugging Face login only if a later cell says the model is gated.


In [ ]:
# Skip unless the model download reports gated/401.
# !uv run hf auth login


In [ ]:
%%writefile /content/select_config.py
"""Copy the committed GPU config to /content/train_run.yaml."""
import shutil
import torch

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print("GPU:", torch.cuda.get_device_name(0), "capability:", (major, minor))
else:
    major = 0
    print("No CUDA GPU visible; using T4 resolved config.")
src = "configs/train_colab.yaml" if major >= 8 else "configs/train_t4_run.yaml"
shutil.copy(src, "/content/train_run.yaml")
print("copied", src, "-> /content/train_run.yaml")


In [ ]:
!uv run python /content/select_config.py


In [ ]:
%cd /content/local-slm-de-address-repair
!uv run python scripts/train_sft.py --config /content/train_run.yaml --dry-run


In [ ]:
!uv run python scripts/train_sft.py --config /content/train_run.yaml
# After a disconnect: redo clone, data, prepare, select_config, then:
# !uv run python scripts/train_sft.py --config /content/train_run.yaml --resume


In [ ]:
# Recovery only: training reached epoch 3 but scoring died.
# The 2026-09-10 T4 run used this path.
# !uv run python scripts/train_sft.py --config /content/train_run.yaml --select-only


In [ ]:
!cat /content/drive/MyDrive/local-slm-de-address-repair/checkpoints/selection.json
!uv run python -c "
import json
base = '/content/drive/MyDrive/local-slm-de-address-repair/checkpoints'
try:
    r = json.load(open(base + '/training_record.json'))
    print({k: r[k] for k in ('winner', 'winner_selection_score', 'train_seconds', 'val_scoring_seconds', 'resumed')})
except FileNotFoundError:
    r = json.load(open(base + '/selection_record.json'))
    print({k: r[k] for k in ('winner', 'winner_selection_score')})
print(r['gpu'])
"


Merge the winner into the pinned MiniCPM5-1B revision, then export GGUF
at llama.cpp `b31b71f3a076bfc4278daad442203a9c51c6e676`. Build with `-j2`.
The T4 winner was `checkpoint-939`; the merge cell reads `selection.json`.


In [ ]:
%%writefile /content/merge_sft.py
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_id, rev = "openbmb/MiniCPM5-1B", "87179e5c1f455ef22e6223592d2d61351b525bfc"
ckpt_root = "/content/drive/MyDrive/local-slm-de-address-repair/checkpoints"
winner = json.load(open(ckpt_root + "/selection.json"))["winner"]
out = "/content/drive/MyDrive/local-slm-de-address-repair/merged-sft"
tok = AutoTokenizer.from_pretrained(base_id, revision=rev)
base = AutoModelForCausalLM.from_pretrained(
    base_id, revision=rev, torch_dtype="auto", device_map="cpu"
)
model = PeftModel.from_pretrained(base, f"{ckpt_root}/{winner}")
merged = model.merge_and_unload()
merged.save_pretrained(out)
tok.save_pretrained(out)
print("merged", winner, "->", out)


In [ ]:
!uv run python /content/merge_sft.py


In [ ]:
!git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp && cd /content/llama.cpp && git checkout b31b71f3a076bfc4278daad442203a9c51c6e676
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF && cmake --build build --config Release -j2 --target llama-quantize


In [ ]:
!uv run --with gguf --with sentencepiece python /content/llama.cpp/convert_hf_to_gguf.py /content/drive/MyDrive/local-slm-de-address-repair/merged-sft --outfile /content/drive/MyDrive/local-slm-de-address-repair/sft-f16.gguf --outtype f16
!/content/llama.cpp/build/bin/llama-quantize /content/drive/MyDrive/local-slm-de-address-repair/sft-f16.gguf /content/drive/MyDrive/local-slm-de-address-repair/sft-Q4_K_M.gguf Q4_K_M
